# Feedback QLoRA: Qwen3.5-9B student

Fine-tunes Qwen3.5-9B with QLoRA to write tutor feedback, using the teacher feedback from `feedback_teacher_qwen3.5-27B.ipynb` as targets.

- **Same base model as the grading adapter** (`qlora/`), so one vLLM server can serve both adapters: grade first, then write feedback from the grades.
- **Same 80/10/10 question split as the grading notebook** (unique stripped questions shuffled with seed 42). Test questions are unseen by both adapters, so the grade-then-feedback pipeline can be evaluated end to end.
- **Input:** question, reference answer, student answer and each key point marked MET / NOT MET. **Target:** the teacher feedback. Only rows that passed the teacher's checks are used for training and validation.
- Loss on the assistant turn only; thinking disabled for training and generation.
- Validation loss is checked four times per epoch; the best adapter is restored at the end.
- After training, the test set is generated with the base model and the adapter, and cheap automatic checks are reported here. The LLM-judge comparison runs in the teacher notebook (last section), because the 27B judge and this model do not fit on the GPU together.

## Install dependencies

Run once, then restart the kernel. Use a separate environment from the vLLM one used by the teacher notebook: these installs can change the `transformers`/`torch` versions vLLM is pinned to.

`flash-linear-attention` provides fast kernels for Qwen3.5's linear-attention layers. They are Triton-based, so no `nvcc` is needed. Without them transformers falls back to a much slower PyTorch path.

In [ ]:
import sys
!{sys.executable} -m pip install -q transformers peft trl datasets accelerate bitsandbytes flash-linear-attention pandas

## Configuration

In [ ]:
from __future__ import annotations

import gc
import importlib.util
import json
import math
import os
import random
import re
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import create_repo, get_token, upload_folder
from IPython.display import display
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, set_seed
from trl import SFTConfig, SFTTrainer

BASE_DIR = Path.cwd()
if BASE_DIR.name != "feedback" and (BASE_DIR / "feedback").is_dir():
    BASE_DIR = BASE_DIR / "feedback"

TEACHER_DIR = BASE_DIR / "Datasets" / "teacher_feedback"
INPUT_PATH = TEACHER_DIR / "training_data_moh_eng_rubrics_with_feedback.csv"
PREDICTIONS_PATH = TEACHER_DIR / "student_test_predictions.jsonl"  # read by the judge section of the teacher notebook
OUTPUT_DIR = BASE_DIR / "qwen35-9b-feedback-lora"

MODEL_ID = "Qwen/Qwen3.5-9B"
RUN_TRAINING = True  # False loads the adapter already saved in OUTPUT_DIR and only evaluates
SEED = 42
SPLIT_SEED = 42  # must match the grading notebook so both adapters share one test set

# r=16 fits the default --max-lora-rank of qlora/serve_qwen35_vllm.sh; style imitation on ~2.8k rows needs little capacity.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch 16
VALIDATIONS_PER_EPOCH = 4
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4

MAX_NEW_TOKENS = 320  # the longest teacher feedback is ~140 words (~200 tokens)
GEN_BATCH_SIZE = 16  # left-padded batched greedy decoding; set 1 if the batch check below disagrees

RUN_HF_UPLOAD = True  # uploads the final adapter after the test set has been generated
HF_REPO_ID = "SmuFypTeam5/feedback"
evaluation_completed = False

set_seed(SEED)
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("flash-linear-attention installed:", importlib.util.find_spec("fla") is not None)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
print("Input:", INPUT_PATH)

## Load the teacher feedback and split by question

The split repeats the grading notebook's code exactly. It is keyed on the raw question text, before any normalisation.

In [ ]:
raw_df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
df = raw_df.copy()
for column in ["question", "desired_answer", "student_answer"]:
    df[column] = df[column].str.replace("\r\n", "\n").str.replace("\r", "\n")
df["criteria"] = df["criteria[]"].map(json.loads)
df["criteria_score"] = df["criteria_score[]"].map(json.loads)
df["criteria_max_score"] = df["criteria_max_score[]"].map(json.loads)
df["passed"] = df["feedback_passed_checks"].str.lower().eq("true")
met_counts = df.apply(lambda row: sum(s >= m for s, m in zip(row["criteria_score"], row["criteria_max_score"])), axis=1)
df["outcome"] = [
    "all met" if met == len(row) else "none met" if met == 0 else "partial"
    for met, row in zip(met_counts, df["criteria"])
]

# Same procedure as qlora/Rubrics: shuffle unique stripped questions, then 80/10/10.
question_keys = [question.strip() for question in raw_df["question"]]
unique_questions = list(dict.fromkeys(question_keys))
random.Random(SPLIT_SEED).shuffle(unique_questions)
train_end = int(len(unique_questions) * 0.80)
validation_end = train_end + int(len(unique_questions) * 0.10)
question_to_split = {
    **{question: "train" for question in unique_questions[:train_end]},
    **{question: "validation" for question in unique_questions[train_end:validation_end]},
    **{question: "test" for question in unique_questions[validation_end:]},
}
df["split"] = [question_to_split[question] for question in question_keys]

print(f"{len(df)} rows, {len(unique_questions)} questions, {df['passed'].sum()} with passed teacher feedback")
print(pd.crosstab(df["split"], df["outcome"], margins=True), "\n")
print(pd.crosstab(df["split"], df["passed"].map({True: "passed", False: "failed"})))

## Prompt

These must match what the backend sends at inference time. `STUDENT_SYSTEM_PROMPT` and `build_row_prompt` are copied from the teacher notebook.

In [ ]:
STUDENT_SYSTEM_PROMPT = (
    "You are a computer science tutor. Write concise, specific, encouraging feedback to the student on their short answer. "
    "The marker's decision for each key point is given; help the student understand it. "
    "Acknowledge what is correct, explain anything missing or incorrect, help the student improve, "
    "and never mention marks, key points or the reference answer."
)


def criterion_marks(row) -> list[bool]:
    return [score >= maximum for score, maximum in zip(row["criteria_score"], row["criteria_max_score"])]


def build_row_prompt(row) -> str:
    marks = criterion_marks(row)
    key_points = [
        f"{number}. {'[MET] ' if met else '[NOT MET] '}{criterion}"
        for number, (criterion, met) in enumerate(zip(row["criteria"], marks), start=1)
    ]
    return "\n\n".join(
        [
            f"Question:\n{row['question'].strip()}",
            f"Reference answer:\n{row['desired_answer'].strip()}",
            f"Student answer:\n{row['student_answer'].strip() or '(no answer given)'}",
            "Key points:\n" + "\n".join(key_points),
            f"Key points met: {sum(marks)} of {len(row['criteria'])}",
        ]
    )


def make_messages(row) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": STUDENT_SYSTEM_PROMPT},
        {"role": "user", "content": build_row_prompt(row)},
    ]


def make_dataset(split: str) -> Dataset:
    rows = df[(df["split"] == split) & df["passed"]]
    return Dataset.from_list(
        [
            {
                "row_index": int(index),
                "prompt": make_messages(row),
                "completion": [{"role": "assistant", "content": row["feedback"]}],
            }
            for index, row in rows.iterrows()
        ]
    )


train_dataset, validation_dataset = make_dataset("train"), make_dataset("validation")
test_df = df[df["split"] == "test"]
print(f"train {len(train_dataset)} | validation {len(validation_dataset)} | test {len(test_df)} rows (all test rows are generated)")
print(train_dataset[0]["prompt"][1]["content"], "\n---\n", train_dataset[0]["completion"][0]["content"])

## Tokenize and mask the prompt

Same approach as the grading notebook. The generation prompt must be an exact prefix of the full conversation. Prompt tokens get label `-100`, so the loss covers only the feedback and its end token. Nothing is truncated: the sequence limit is measured from the data.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def tokenize_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False, add_generation_prompt=True, enable_thinking=False
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


formatted_train, formatted_validation = (
    split.map(tokenize_example, remove_columns=split.column_names, desc=f"Tokenize {name}")
    for name, split in (("train", train_dataset), ("validation", validation_dataset))
)
lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
sequence_limit = math.ceil(max(lengths) / 32) * 32
print("Token lengths:", pd.Series(lengths).describe().round(0).to_dict())
print("Sequence limit:", sequence_limit)
print("Supervised part of example 0:", tokenizer.decode([t for t in formatted_train[0]["labels"] if t != -100]))

## Load the 4-bit model and attach LoRA

NF4 with double quantisation and BF16 compute. LoRA targets the attention and MLP projections of the text layers, the same selection as the grading adapter. With `RUN_TRAINING = False`, the saved adapter is loaded instead.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA training requires a CUDA GPU.")
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    ),
    device_map={"": 0},
    dtype=compute_dtype,
    attn_implementation="sdpa",
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

if RUN_TRAINING:
    model.config.use_cache = False
    for param in model.parameters():
        param.requires_grad_(False)
    projection_names = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    target_modules = [
        name for name, module in model.named_modules()
        if name.startswith("model.layers.") and isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in projection_names
    ]
    if not target_modules:
        raise RuntimeError("No text projection layers found; inspect model.named_modules().")
    print("LoRA projection layers:", len(target_modules))
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
    ))
    model.print_trainable_parameters()
else:
    model = PeftModel.from_pretrained(model, OUTPUT_DIR)
    print("Loaded adapter from", OUTPUT_DIR)

## Train

Paged 8-bit AdamW and gradient checkpointing. Validation loss is checked about four times per epoch. The learning rate halves after two checks without improvement, training stops after four, and the lowest-validation-loss checkpoint is restored.

In [ ]:
steps_per_epoch = math.ceil(len(formatted_train) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
eval_steps = max(1, round(steps_per_epoch / VALIDATIONS_PER_EPOCH))
print(f"{steps_per_epoch} optimizer steps per epoch; validating every {eval_steps}")

if RUN_TRAINING:
    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR),
            seed=SEED,
            data_seed=SEED,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=TRAIN_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            optim="paged_adamw_8bit",
            lr_scheduler_type="reduce_lr_on_plateau",
            lr_scheduler_kwargs={
                "mode": "min", "factor": LR_REDUCTION_FACTOR, "patience": LR_PATIENCE, "min_lr": MIN_LEARNING_RATE,
            },
            bf16=(compute_dtype == torch.bfloat16),
            fp16=(compute_dtype == torch.float16),
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            loss_type="nll",
            completion_only_loss=True,
            max_length=sequence_limit,
            packing=False,
            padding_free=False,
            logging_steps=max(1, eval_steps // 2),
            save_strategy="steps",
            save_steps=eval_steps,
            eval_strategy="steps",
            eval_steps=eval_steps,
            prediction_loss_only=True,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=3,
            report_to="none",
        ),
        train_dataset=formatted_train,
        eval_dataset=formatted_validation,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    # The collator must keep prompt and padding labels at -100 and the feedback supervised.
    samples = [trainer.train_dataset[i] for i in range(2)]
    batch = trainer.data_collator(samples)
    for index, sample in enumerate(samples):
        labels, length = batch["labels"][index], len(sample["labels"])
        assert labels[:length].tolist() == sample["labels"], "Collator changed the completion mask."
        assert (labels[length:] == -100).all(), "Padding contributes to the loss."
    print("Loss masks verified.")

In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print("Saved best adapter to", OUTPUT_DIR)
    display(train_result.metrics)
    history = pd.DataFrame(trainer.state.log_history)
    if "eval_loss" in history:
        display(history.dropna(subset=["eval_loss"])[["step", "epoch", "eval_loss"]])

    # Free optimizer state before generation.
    model = trainer.model
    model.zero_grad(set_to_none=True)
    trainer.optimizer = None
    trainer.lr_scheduler = None
    trainer.accelerator.free_memory()
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

## Generation and automatic checks

Greedy decoding, matching the backend's `LLM_TEMPERATURE=0`. The checks repeat the teacher's rule checks: grading language, markdown, non-English text, truncation and length. ROUGE-L against the teacher feedback is a rough similarity score; the LLM judge is the real quality measure.

In [ ]:
GRADING_TERMS = re.compile(
    r"\b(criteri(on|a)|rubric|checklist|key points|not met|scor(e|es|ed|ing)|grad(e|ed|ing)|marker|"
    r"(full|no|lose|lost|earn|earned|award|awarded) marks?|(reference|model|desired|expected|sample) answer)\b",
    re.IGNORECASE,
)
MARKDOWN = re.compile(r"\*\*|__|^\s*#|^\s*[-*•]\s|^\s*\d+[.)]\s", re.MULTILINE)
NON_ENGLISH = re.compile(r"[　-鿿가-힯]")
EOS_IDS = set(model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, list)
              else [model.generation_config.eos_token_id])


def format_issues(text: str, truncated: bool) -> list[str]:
    issues = []
    words = len(text.split())
    if truncated:
        issues.append("truncated")
    if words < 15:
        issues.append("too short")
    if words > 200:
        issues.append("too long")
    if GRADING_TERMS.search(text):
        issues.append("grading language")
    if MARKDOWN.search(text):
        issues.append("markdown")
    if NON_ENGLISH.search(text):
        issues.append("non-English")
    return issues


def rouge_l_f1(candidate: str, reference: str) -> float:
    a, b = re.findall(r"\w+", candidate.lower()), re.findall(r"\w+", reference.lower())
    if not a or not b:
        return 0.0
    previous = [0] * (len(b) + 1)
    for token in a:
        current = [0]
        for j, other in enumerate(b, start=1):
            current.append(previous[j - 1] + 1 if token == other else max(previous[j], current[j - 1]))
        previous = current
    lcs = previous[-1]
    return 0.0 if lcs == 0 else 2 * lcs / (len(a) + len(b))


@torch.inference_mode()
def generate_feedback(rows: pd.DataFrame, batch_size: int = GEN_BATCH_SIZE) -> list[dict]:
    tokenizer.padding_side = "left"  # decoder-only batching pads on the left
    device = model.get_input_embeddings().weight.device
    results = []
    try:
        for start in range(0, len(rows), batch_size):
            batch = rows.iloc[start : start + batch_size]
            texts = [
                tokenizer.apply_chat_template(make_messages(row), tokenize=False, add_generation_prompt=True, enable_thinking=False)
                for _, row in batch.iterrows()
            ]
            encoded = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            generated = model.generate(**encoded, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
            for index, ids in zip(batch.index, generated[:, encoded["input_ids"].shape[1] :].tolist()):
                text = tokenizer.decode(ids, skip_special_tokens=True).split("</think>")[-1].strip()
                results.append({"row_index": int(index), "feedback": text, "truncated": not EOS_IDS & set(ids)})
            print(f"  {min(start + batch_size, len(rows))}/{len(rows)}", end="\r")
    finally:
        tokenizer.padding_side = "right"
    return results

### Batched-generation check

Left-padded batches should match one-at-a-time decoding. Tiny numerical differences can change a word or two late in a reply. If the batched outputs are garbled or very different, set `GEN_BATCH_SIZE = 1`.

In [ ]:
check_rows = test_df.head(4)
single = [generate_feedback(check_rows.iloc[[i]], batch_size=1)[0]["feedback"] for i in range(len(check_rows))]
batched = [item["feedback"] for item in generate_feedback(check_rows, batch_size=len(check_rows))]
for one, many in zip(single, batched):
    print(f"ROUGE-L single vs batched: {rouge_l_f1(one, many):.2f}")
print("\nSingle:\n", single[0], "\n\nBatched:\n", batched[0])

## Generate the test set: base model vs adapter

All 389 test rows, including the 24 whose teacher feedback failed. Their inputs are still valid; they just have no teacher reference for ROUGE-L. The base model is the same weights with the adapter disabled, the same prompt and the same decoding.

In [ ]:
print("Fine-tuned adapter")
predictions = [{"model": "qlora", **item} for item in generate_feedback(test_df)]
print("\nBase model (adapter disabled)")
with model.disable_adapter():
    predictions += [{"model": "base", **item} for item in generate_feedback(test_df)]

with PREDICTIONS_PATH.open("w", encoding="utf-8") as handle:
    for item in predictions:
        handle.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"\nWrote {len(predictions)} predictions to {PREDICTIONS_PATH}")

In [ ]:
evaluation = pd.DataFrame(predictions)
evaluation["outcome"] = evaluation["row_index"].map(df["outcome"])
evaluation["words"] = evaluation["feedback"].str.split().str.len()
evaluation["issues"] = [format_issues(text, truncated) for text, truncated in zip(evaluation["feedback"], evaluation["truncated"])]
evaluation["clean"] = evaluation["issues"].str.len().eq(0)
teacher = df.loc[evaluation["row_index"], ["feedback", "passed"]].reset_index(drop=True)
evaluation["rouge_l"] = [
    rouge_l_f1(text, reference) if passed else float("nan")
    for text, reference, passed in zip(evaluation["feedback"], teacher["feedback"], teacher["passed"])
]

summary = evaluation.groupby("model").agg(
    rows=("row_index", "size"),
    clean_rate=("clean", "mean"),
    mean_words=("words", "mean"),
    rouge_l_vs_teacher=("rouge_l", "mean"),
)
issue_rates = evaluation.explode("issues").dropna(subset=["issues"]).groupby(["model", "issues"]).size().unstack(fill_value=0)
display(summary.join(issue_rates.div(summary["rows"], axis=0).add_prefix("rate: ")).round(3))
print("Mean words by outcome (teacher targets: ~30-60 when all met, ~50-130 otherwise)")
display(evaluation.pivot_table(index="outcome", columns="model", values="words", aggfunc="mean").round(0))
evaluation_completed = True

In [ ]:
# Side by side on a few test rows from each outcome.
for _, sample in test_df.groupby("outcome").sample(n=2, random_state=SEED).iterrows():
    print("=" * 100)
    print(build_row_prompt(sample))
    for name in ["base", "qlora"]:
        text = evaluation[(evaluation["model"] == name) & (evaluation["row_index"] == sample.name)]["feedback"].iloc[0]
        print(f"\n--- {name} ---\n{text}")
    print(f"\n--- teacher ({'passed' if sample['passed'] else 'failed checks'}) ---\n{sample['feedback']}")

## Upload the adapter to Hugging Face

Runs only when `RUN_HF_UPLOAD = True` and the test set has been generated and checked above.
- It uploads `OUTPUT_DIR`: the best adapter (`adapter_config.json`, `adapter_model.safetensors`) and the tokenizer. Intermediate `checkpoint-*` folders are left out.
- It needs a Hugging Face token with write access to the `SmuFypTeam5` organisation. Provide it by exporting `HF_TOKEN` or by running `hf auth login` once on the cluster. The cell stops with a clear error if neither is set.
- If the repository doesn't exist yet, it is created as private. An existing repository keeps its current visibility.

In [ ]:
if RUN_HF_UPLOAD:
    if not evaluation_completed:
        raise RuntimeError("Generate and check the test set above before uploading.")
    missing = [name for name in ("adapter_config.json", "adapter_model.safetensors") if not (OUTPUT_DIR / name).is_file()]
    if missing:
        raise RuntimeError(f"Adapter files missing from {OUTPUT_DIR}: {missing}")
    # HF_TOKEN or a cached `hf auth login`. Fail early: papermill cannot answer an interactive login prompt.
    token = get_token()
    if not token:
        raise RuntimeError("No Hugging Face token: export HF_TOKEN=hf_... or run `hf auth login` on the cluster first.")
    create_repo(HF_REPO_ID, repo_type="model", private=True, exist_ok=True, token=token)
    upload_result = upload_folder(
        token=token,
        folder_path=str(OUTPUT_DIR),
        repo_id=HF_REPO_ID,
        repo_type="model",
        ignore_patterns=["checkpoint-*"],
        commit_message="Upload Qwen3.5-9B feedback QLoRA adapter",
    )
    print("Uploaded adapter to:", upload_result)
else:
    print("Hugging Face upload skipped. Set RUN_HF_UPLOAD=True to publish.")

## Next steps

1. **Judge the test outputs.** In the teacher notebook, run the setup cells and the judge-model cells (through the rule-check cell), then the last section, *Judge the student model*. It scores `student_test_predictions.jsonl` with the same four checks the teacher's own feedback had to pass, for base vs adapter. This is the headline result: judge pass rate by model and outcome.
2. **Serve it.** In `qlora/serve_qwen35_vllm.sh`, add a second `--lora-modules` entry, e.g. `qwen35-9b-feedback-qlora=SmuFypTeam5/feedback` (or the local `qwen35-9b-feedback-lora` folder). `--max-lora-rank` must be at least the larger of the two adapters' ranks. One base model then serves grading and feedback.
3. **Backend.** `grading_service.py` currently asks one model for the grades and the feedback in one JSON reply. With the adapters, that becomes two calls:
   - Call the grading adapter for the MET/NOT MET array.
   - Build this notebook's prompt from those marks and call the feedback adapter with `STUDENT_SYSTEM_PROMPT`, `enable_thinking: false` and `temperature: 0`.

   The prompt text must match `build_row_prompt` exactly.
4. **Optional end-to-end check.** Here the marks are the dataset's gold marks. In production they come from the grading adapter, which is sometimes wrong. Re-running the test generation with the grading adapter's predicted marks shows how much grading errors hurt the feedback.